# Capacity Test with CapTest Class

This example shows use of the CapTest class to conduct a capacity test. The CapTest class provides an entry point to define the type of test (regression formula) and define other test level parameters like the test tolerance and reporting conditions.

This example performs the same test as the Complete Capacity Testing example, but uses the `run_test` function to apply the filters and then displays the filtering steps visually using the `scatter_filters` method.

## Imports

In [ ]:
import inspect
import warnings
# warnings.filterwarnings('ignore')
from IPython.display import display, Markdown

import pandas as pd

# import captest as pvc
import captest as ct
from captest import capdata as pvc
from captest.captest import CapTest
from bokeh.io import output_notebook, show

# uncomment below two lines to use cptest.scatter_hv in notebook
import holoviews as hv
hv.extension('bokeh')

#if working offline with the CapData.plot() method may fail
#run 'export BOKEH_RESOURCES=inline' at the command line before
#running the jupyter notebook

output_notebook()

## Load and Plot Measured Data
Load the measured data with the `load_data` method, which returns a CapData object. This example uses a column grouping defined in an excel file. 

In [ ]:
## uncomment line below to review options for pre-defined capacity tests
# ct.captest.TEST_SETUPS.keys()

In [ ]:
## uncomment line below to review the full docstring with all parameters for the CapTest class
# display(Markdown(f"```\n{inspect.getdoc(CapTest)}\n```"))

In [ ]:
ts = CapTest.from_params(
    test_setup='e2848_default',
    meas_path='./data/example_meas_data.csv',
    sim_path='./data/pvsyst_example_HourlyRes_2.CSV',
    test_tolerance='+/- 3',
    ac_nameplate=6_000,
    meas_load_kwargs={
        'group_columns': './data/column_groups.xlsx',
    }
)

In [ ]:
ts.meas.agg_sensors(agg_map={'real_pwr_inv':'sum'}, verbose=True)

Note, the full functionality of the dashboard requires a live notebook. Try installing to run or using the launch binder button at the top of the page. 

In [ ]:
combine = {'inv_sum_mtr_pwr': ['mtr', 'inv.*agg'], 'irr_all':['irr_poa', 'irr_ghi'], 'temp_all':['temp_amb', 'temp_mod']}
default_groups = ['inv_sum_mtr_pwr', 'irr_all', 'temp_all']
ts.meas.plot(combine=combine, default_groups=default_groups)

## Filtering Measured Data
The `CapData` class provides a number of convience methods to apply filtering steps as defined in ASTM E2848.  The following section demonstrates the use of the more commonly used filtering steps to remove measured data points.

In [ ]:
# Uncomment and run to copy over the filtered dataset with the unfiltered data.
ts.meas.reset_filter()

In [ ]:
measured_filters = [
    (pvc.CapData.filter_custom, (pd.DataFrame.dropna, ), {}),
    (pvc.CapData.filter_irr, (ts.min_irr, ts.max_irr), {}),
    (pvc.CapData.filter_outliers, (), {}),
    (pvc.CapData.fit_regression, (), {'filter':True, 'summary':False}),
    (pvc.CapData.rep_cond, (), {}),
    (pvc.CapData.filter_irr, (ts.rep_irr_filter_low, ts.rep_irr_filter_high), {'ref_val':'rep_irr'}),
    (pvc.CapData.fit_regression, (), {}),
]

In [ ]:
pvc.run_test(ts.meas, measured_filters)

In [ ]:
ts.meas.get_summary()

In [ ]:
import panel as pn

In [ ]:
# sp = ct.plotting.ScatterPlot(cd=ts.meas, timeseries=True)

In [ ]:
# dboard = pn.Row(sp.param, sp.view)
# pn.serve(dboard)

In [ ]:
(
    ts.meas.scatter_filters().opts(tools=['lasso_select']) +
    ts.meas.timeseries_filters().opts(width=1000)
)

In [ ]:
ct.plotting.ScatterPlot(cd=ts.meas, timeseries=True, tc_power=True, tc_mode='replace', split_day=True).view()

## Load and Filter PVsyst Data

To load and filter the modeled data, often from PVsyst, we simply create a new CapData object, load the PVsyst data, and apply the filtering methods as appropriate.

To load pvsyst data we use the `load_data` method with the `load_pvsyst` option set to True.  By default the `load_data` method will search for a csv file that includes `pvsyst` in the filename in a `data` directory in the same directory as this file.  If you have saved the pvsyst file in a different location, you can use the `path` and `fname` arguments to load it.

In [ ]:
# Write over cptest.flt_sim dataframe with a copy of the original unfiltered dataframe
ts.sim.reset_filter()

In [ ]:
ts.min_irr

In [ ]:
simulated_filters = [
    (pvc.CapData.filter_time, (), {'test_date':'10/11/1990', 'days':ts.sim_days}),
    (pvc.CapData.filter_irr, (ts.min_irr, 930), {}),
    (pvc.CapData.filter_pvsyst, (), {}),
    (pvc.CapData.filter_irr, (ts.rep_irr_filter_low, ts.rep_irr_filter_high), {'ref_val':ts.meas.rc['poa'][0]}),
    (pvc.CapData.fit_regression, (), {}),
]

In [ ]:
pvc.run_test(ts.sim, simulated_filters)

In [ ]:
ts.sim.get_summary()

In [ ]:
ts.sim.scatter_filters()

## Results

The `get_summary` and `captest_results_check_pvalues` functions display the results of filtering on simulated and measured data and the final capacity test results comparing measured capacity to expected capacity, respectively.

In [ ]:
ts.get_summary()

In [ ]:
ts.meas.rc

In [ ]:
ts.meas.print_points_summary(ts.hrs_req)

In [ ]:
ts.captest_results_check_pvalues()

The `overlay_scatters` function can be used to overlay the final scatter plots from scatter plots of all filtering steps produced above.

In [ ]:
ts.overlay_scatters()